In [11]:
import numpy as np
import pandas as pd


In [36]:
dtype_map = {
    "Task Id": "string",
    "FPOGV": "Int64", "FPOGID": "Int64",
    "LPMMV": "Int64", "RPMMV": "Int64",
    "BKID": "Int64",  "BKDUR": "float64", 
    "AOI": "string"
}
fixations_onoff = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/fix_onoff_task1.csv", dtype=dtype_map)
gaze_onoff = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/gaze_onoff_task1.csv", dtype=dtype_map)

In [37]:

TICK_HZ = 10_000_000  # TIMETICK(f=10000000)

# --- Build ONE shared anchor per (Video, User) across any number of tables ---
def build_shared_anchor(*dfs,
                        tick_col: str = "TIMETICK(f=10000000)",
                        group_cols=("Video Name (folder)", "User ID (file)")) -> pd.DataFrame:
    refs = []
    for d in dfs:
        tmp = d[list(group_cols) + [tick_col]].copy()
        tmp[tick_col] = pd.to_numeric(tmp[tick_col], errors="coerce")
        refs.append(tmp.groupby(list(group_cols), as_index=False)[tick_col].min()
                      .rename(columns={tick_col: "min_tick"}))
    ref = (pd.concat(refs, ignore_index=True)
             .groupby(list(group_cols), as_index=False)["min_tick"].min()
             .rename(columns={"min_tick": "anchor_tick"}))
    return ref

# 1) Window ids by TIMETICK (per Video+User), no task-start anchoring
def add_window_id_by_timetick(
    df: pd.DataFrame,
    *,
    win_sec: float,
    tick_col: str = "TIMETICK(f=10000000)",
    group_cols=("Video Name (folder)", "User ID (file)"),
    shared_ref: pd.DataFrame | None = None,
    out_col: str = "win_id",
) -> pd.DataFrame:
    out = df.copy()
    out[tick_col] = pd.to_numeric(out[tick_col], errors="coerce")
    win_ticks = int(round(win_sec * TICK_HZ))

    if shared_ref is not None:
        out = out.merge(shared_ref, on=list(group_cols), how="left", validate="many_to_one")
        anchor = out["anchor_tick"]
    else:
        anchor = out.groupby(list(group_cols))[tick_col].transform("min")

    out[out_col] = ((out[tick_col] - anchor) // win_ticks).astype("Int64")
    return out

# 1b) Refine window labels by majority-fraction (label denoising)
def refine_window_labels_by_fraction(
    df_with_win: pd.DataFrame,
    *,
    group_cols=("Video Name (folder)", "User ID (file)"),
    win_col="win_id",
    label_col="label",
    on_hi=0.80,
    off_lo=0.20,
    out_col="label",
) -> pd.DataFrame:
    d = df_with_win.copy()
    d[label_col] = d[label_col].astype(str).str.upper().str.strip()
    d = d[d[label_col].isin(["ON","OFF"])].copy()

    key = list(group_cols) + [win_col]
    d["_is_on"] = (d[label_col] == "ON").astype(float)

    frac = (d.groupby(key, dropna=False)["_is_on"]
              .mean()
              .rename("on_fraction")
              .reset_index())

    frac["label_refined"] = np.where(frac["on_fraction"] >= on_hi, "ON",
                              np.where(frac["on_fraction"] <= off_lo, "OFF", np.nan))

    d = d.merge(frac[key + ["label_refined","on_fraction"]],
                on=key, how="left", validate="many_to_one")

    d = d[~d["label_refined"].isna()].copy()
    d[out_col] = d["label_refined"]
    d.drop(columns=["_is_on","label_refined"], inplace=True)
    return d

# ---- small helpers for robust stats inside windows ----
def _mad(x):
    x = pd.Series(x, dtype="float64").dropna()
    if x.empty: return np.nan
    med = x.median()
    return (x - med).abs().median()

def _pupil_slope(tick, pupil):
    # robust slope via np.polyfit on valid points; time in seconds from first sample
    if tick is None:
        return np.nan
    tt = pd.Series(tick, dtype="float64")
    pp = pd.Series(pupil, dtype="float64")
    m = (~tt.isna()) & (~pp.isna())
    if m.sum() < 5:
        return np.nan
    tsec = (tt[m] - tt[m].iloc[0]) / TICK_HZ
    try:
        slope, _ = np.polyfit(tsec.values, pp[m].values, 1)
    except Exception:
        slope = np.nan
    return slope

def _circular_variance_deg(angles_deg):
    a = pd.Series(angles_deg, dtype="float64").dropna()
    if a.empty: return np.nan
    rad = np.deg2rad(a.values)
    C = np.mean(np.cos(rad))
    S = np.mean(np.sin(rad))
    R = np.sqrt(C*C + S*S)
    return 1 - R  # 0=tightly clustered, 1=uniform

def _gaze_entropy_2d(x, y, *, bins=8, min_valid=100):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < min_valid or y.size < min_valid:
        return np.nan
    x = np.clip(x, 0.0, 1.0)
    y = np.clip(y, 0.0, 1.0)
    H, _, _ = np.histogram2d(x, y, bins=bins, range=[[0,1],[0,1]], density=False)
    total = H.sum()
    if total <= 0:
        return np.nan
    p = (H / total).ravel()
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

# 2) FIXATION features → cleaned + robust features
def window_features_fixations(fix_df: pd.DataFrame, *, win_sec: float) -> pd.DataFrame:
    keys = ["Video Name (folder)","User ID (file)","Task Id","type","label","phase","block_id","win_id"]
    eps = 1e-6

    def _agg(w: pd.DataFrame) -> pd.Series:
        fix_count     = w["FPOGID"].nunique() if "FPOGID" in w else np.nan
        fix_mean_dur  = w["FPOGD"].mean()     if "FPOGD"  in w else np.nan
        fix_median_dur= w["FPOGD"].median()   if "FPOGD"  in w else np.nan
        fix_aoi_ratio = w["AOI"].notna().mean() if "AOI" in w else np.nan

        fix_efficiency = (fix_count / (abs(fix_mean_dur) + eps)) if pd.notna(fix_mean_dur) else np.nan
        fix_rate_per_s = (fix_count / win_sec) if pd.notna(fix_count) else np.nan

        return pd.Series({
            "fix_efficiency":  fix_efficiency,
            "fix_aoi_ratio":   fix_aoi_ratio,
            "fix_rate_per_s":  fix_rate_per_s,
            "fix_median_dur":  fix_median_dur,
        })

    return (fix_df.dropna(subset=["win_id"])
                 .groupby(keys, dropna=False)
                 .apply(_agg)
                 .reset_index())

# 3) GAZE features → cleaned + robust features
def window_features_gaze(gaze_df: pd.DataFrame, *, win_sec: float = 3.0) -> pd.DataFrame:
    keys = ["Video Name (folder)","User ID (file)","Task Id","type","label","phase","block_id","win_id"]

    def _agg(w: pd.DataFrame) -> pd.Series:
        gv = (w["FPOGV"] == 1) if "FPOGV" in w else pd.Series(True, index=w.index)
        lv = (w["LPMMV"] == 1) if "LPMMV" in w else pd.Series(False, index=w.index)
        rv = (w["RPMMV"] == 1) if "RPMMV" in w else pd.Series(False, index=w.index)

        # --- dispersion (std-based) ---
        gx = w.loc[gv, "FPOGX"].std() if "FPOGX" in w else np.nan
        gy = w.loc[gv, "FPOGY"].std() if "FPOGY" in w else np.nan
        gaze_disp_total = (np.sqrt((0 if pd.isna(gx) else gx**2) + (0 if pd.isna(gy) else gy**2))
                           if (not pd.isna(gx) or not pd.isna(gy)) else np.nan)

        # --- pupil summary ---
        if "LPMM" in w and "RPMM" in w:
            l = w["LPMM"].where(lv)
            r = w["RPMM"].where(rv)
            pupil_row = pd.concat([l, r], axis=1).mean(axis=1, skipna=True)
            pupil_mean  = pupil_row.mean()
            pupil_var   = pupil_row.var(ddof=1)
            pupil_mad   = _mad(pupil_row)
            tick_col = "TIMETICK(f=10000000)"
            pupil_slope = _pupil_slope(w[tick_col] if tick_col in w else None, pupil_row)
        else:
            pupil_mean = pupil_var = pupil_mad = pupil_slope = np.nan

        # --- saccades ---
        saccade_mean_mag = w.loc[gv, "SACCADE_MAG"].mean() if "SACCADE_MAG" in w else np.nan
        if "SACCADE_DIR" in w:
            saccade_circ_var = _circular_variance_deg(w.loc[gv, "SACCADE_DIR"])
            saccade_dir_var  = w.loc[gv, "SACCADE_DIR"].var(ddof=1)
        else:
            saccade_circ_var = np.nan
            saccade_dir_var  = np.nan
        if pd.isna(saccade_dir_var):
            saccade_dir_var = 0.0  # keep old behavior

        # --- blinks & quality ---
        blink_count      = w.loc[gv, "BKID"].nunique() if "BKID" in w else np.nan
        blink_rate       = (blink_count / win_sec) if pd.notna(blink_count) else np.nan
        valid_gaze_ratio = gv.mean() if len(w) else np.nan


        return pd.Series({
            "gaze_dispersion_total": gaze_disp_total,
            "pupil_mean":            pupil_mean,
            "pupil_var":             0 if pd.isna(pupil_var) else pupil_var,
            "pupil_mad":             pupil_mad,
            "pupil_slope":           pupil_slope,
            "saccade_mean_mag":      saccade_mean_mag,
            "saccade_dir_var":       0 if pd.isna(saccade_dir_var) else saccade_dir_var,
            "saccade_circ_var":      saccade_circ_var,
            "blink_rate":            blink_rate,
            "valid_gaze_ratio":      valid_gaze_ratio,
        })

    return (gaze_df.dropna(subset=["win_id"])
                 .groupby(keys, dropna=False)
                 .apply(_agg)
                 .reset_index())

# 4) Merge fixation + gaze features (keep ON/OFF/phase separate)
def combine_fix_gaze_windows(fix_w: pd.DataFrame, gaze_w: pd.DataFrame) -> pd.DataFrame:
    keys = ["Video Name (folder)","User ID (file)","Task Id","type","label","phase","block_id","win_id"]
    merged = pd.merge(fix_w, gaze_w, on=keys, how="outer", validate="one_to_one")
    return merged

In [38]:
# after you add win_id and refine labels for Task 2:
gaze_wid_t2 = add_window_id_by_timetick(gaze_onoff, win_sec=3.0)
# use the Task-2 gaze features:
gaz_w_t2 = window_features_gaze_task2(gaze_wid_t2, win_sec=3.0, min_valid_frac=0.60)
fix_wid_t2 = add_window_id_by_timetick(fixations_onoff, win_sec=3.0)
fix_w_t2 = window_features_fixations(fix_wid_t2, win_sec=3.0)
win_task2 = combine_fix_gaze_windows(fix_w_t2, gaz_w_t2)

/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_4295/4073869376.py:208: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)
/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_4295/1410866294.py:175: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg_task2)
/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_4295/4073869376.py:145: DeprecationWarning: DataFrameGroupBy.

In [39]:
win_task2.to_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/improving models/outputs2/fix_wid_t2.csv", index=False)    